<a href="https://colab.research.google.com/github/vifirsanova/ML-2026-pt-2/blob/main/transformers/transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Language Identification на датасете `papluca/language-identification`

Многоклассовая задача с сильным дисбалансом

**Датасет:** `papluca/language-identification` — 90k текстов на 45 языках (train/validation/test).

**Модель:** `distilbert-base-multilingual-cased` (мультиязычная, чтобы покрыть все 45 классов).

## 1. Установка и импорты

In [ ]:
!pip install -q transformers datasets evaluate accelerate scikit-learn seaborn matplotlib

import torch
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
)
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

print(f"Transformers version: {__import__('transformers').__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 2. Загрузка датасета

In [ ]:
# Загружаем датасет определения языка (45 классов)
raw_datasets = load_dataset("papluca/language-identification")
print(raw_datasets)

# Посмотрим пример
print("\nПример записи:")
print(raw_datasets["train"][0])

# Получаем список классов
label_names = raw_datasets["train"].features["labels"].names
num_labels = len(label_names)
print(f"\nКоличество классов: {num_labels}")
print(f"Классы: {label_names[:10]}... (первые 10)")

## 3. Токенизация

In [ ]:
# Мультиязычная модель для покрытия всех 45 языков
checkpoint = "distilbert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=128,
    )

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
print("Ключи после токенизации:", list(tokenized_datasets["train"][0].keys()))
print(f"Размер train: {len(tokenized_datasets['train'])}")
print(f"Размер validation: {len(tokenized_datasets['validation'])}")
print(f"Размер test: {len(tokenized_datasets['test'])}")

## 4. DataCollator и метрики

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    acc = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
    f1_macro = f1_metric.compute(
        predictions=predictions,
        references=labels,
        average="macro"
    )["f1"]

    return {"accuracy": acc, "f1_macro": f1_macro}

print("compute_metrics готова")

## 5. Модель и Trainer

In [ ]:
id2label = {i: name for i, name in enumerate(label_names)}
label2id = {name: i for i, name in id2label.items()}

model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)

training_args = TrainingArguments(
    output_dir="./lang-id-output",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=100,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,  # v5: processing_class вместо tokenizer
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Trainer готов к обучению")

## 6. Обучение

**Время:** ~15-20 минут на T4 GPU (3 эпохи, 90k примеров train).

Если Colab отключается — уменьшите `num_train_epochs` до 2 или возьмите подвыборку.

In [ ]:
train_result = trainer.train()
print("\nОбучение завершено")
print(f"Train loss: {train_result.training_loss:.4f}")

## 7. Финальные метрики на test

In [ ]:
results = trainer.evaluate(tokenized_datasets["test"])
print("=== Финальные метрики на test ===")
print(f"eval_loss:     {results['eval_loss']:.4f}")
print(f"eval_accuracy: {results['eval_accuracy']:.4f}")
print(f"eval_f1_macro: {results['eval_f1_macro']:.4f}")

gap = results['eval_accuracy'] - results['eval_f1_macro']
print(f"\nРазрыв accuracy - f1_macro: {gap:.4f}")
if gap > 0.05:
    print("Заметный разрыв — модель смещена к частым классам. Смотрите confusion matrix.")

## 8. Confusion Matrix

In [ ]:
predictions = trainer.predict(tokenized_datasets["test"])
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

# Матрица 45x45 — большая. Визуализируем, но для анализа будем использовать top confused pairs.
cm = confusion_matrix(labels, preds)
print(f"Confusion matrix shape: {cm.shape}")

# Найдём топ-10 самых частых ошибок (вне диагонали)
error_pairs = []
for i in range(num_labels):
    for j in range(num_labels):
        if i != j and cm[i, j] > 0:
            error_pairs.append((cm[i, j], i, j))

error_pairs.sort(reverse=True)
print("\nТоп-10 confused pairs")
for count, true_idx, pred_idx in error_pairs[:10]:
    print(f"{label_names[true_idx]:5s} → {label_names[pred_idx]:5s} : {count} ошибок")

## 9. Анализ 5 ошибочных примеров

In [ ]:
error_indices = np.where(preds != labels)[0]
print(f"Всего ошибок: {len(error_indices)} из {len(labels)} ({len(error_indices)/len(labels)*100:.2f}%)")

np.random.seed(42)
sampled = np.random.choice(error_indices, size=min(5, len(error_indices)), replace=False)

print("\n5 случайных ошибочных примеров\n")
for idx in sampled:
    text = raw_datasets["test"][int(idx)]["text"]
    true_label = label_names[labels[idx]]
    pred_label = label_names[preds[idx]]

    print(f"Текст (первые 100 символов): {text[:100]}...")
    print(f"Истинный класс: {true_label} | Предсказано: {pred_label}")

## 10. Гипотезы о причинах ошибок (заполните сами)

На основе вывода выше заполните таблицу для отчёта:

| Текст (первые 100 символов) | Истинный класс | Предсказанный класс | Гипотеза |
|---|---|---|---|
| ... | ... | ... | ... |

**Примеры конкретных гипотез для language identification:**
- «Короткий текст (< 10 символов) — недостаточно контекста для определения языка»
- «Близкородственные языки (например, испанский / португальский) — лексическое пересечение»
- «Латиница с диакритикой, которую токенизатор мультиязычной модели обрабатывает неоптимально»
- «Класс minority (редкий язык) — модель смещена к majority (английский / испанский)»
- «Текст содержит заимствования или имена собственные, не характерные для языка»

## 11. (Опционально) Сохранение модели

In [ ]:
# trainer.save_model("./lang-id-final")
# tokenizer.save_pretrained("./lang-id-final")
# print("Модель сохранена в ./lang-id-final")

---

## Чек-лист для сдачи

- [ ] Запущен `trainer.train()` — модель обучена
- [ ] Получены `eval_accuracy` и `eval_f1_macro` на test
- [ ] Построена confusion matrix (или топ confused pairs)
- [ ] Выбраны 5 **реальных** ошибочных примеров
- [ ] Для каждого написана **конкретная** гипотеза (не «модель не поняла», а «короткий текст», «близкие языки», «редкий класс»)
- [ ] В отчёте указано, какие классы путаются чаще всего

**Почему эта задача сложнее стандартных:** 45 классов, сильный дисбаланс (английский доминирует), близкородственные языки (испанский/португальский/итальянский), короткие тексты. `f1_macro` ожидается заметно ниже `accuracy`.